This notebooks is based on https://www.kaggle.com/code/kawchar85/lb-0-289-improved-basline-prompt-engineering. <br />
What I changed:  <br />
    1.Used multiprocessing to speed up PDF text extraction. <br />
    2.Merged regex patterns into one for faster matching. <br />
    3.Added prompt caching to avoid repeated LLM calls. <br />
    4.Improved DOI parsing with stricter regex. <br />

In [ ]:
import os
import re
import fitz
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from multiprocessing import Pool
from hashlib import md5
import torch
import vllm
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor

text_span_len = 200
NUM_PROCESSES = 8
PROMPT_MAX_TOKENS = 512

# REGEX
re_doi = re.compile(r"10\.\d{4}")
big_re = re.compile(r"GSE\d+|SR[APRX]\d+|PRJ[NAED][A-Z]?\d+|IPR\d{6}|PF\d{5}|EMPIAR-\d{5}|CHEMBL\d+|CVCL_[A-Z0-9]{4}|ENS[A-Z]{0,6}[GT]\d{11}|N[MC]_\d+(?:\.\d+)?|rs\d+|EPI(?:_ISL_)?\d+|PXD\d{6}|SAM[ND]\d+|ERR\d+", re.I)

# REMOVE REFERENCES
def remove_references_section(text):
    lines = text.split('\n')
    cut_index = -1
    for i in range(len(lines) - 1, max(0, int(len(lines) * 0.3)), -1):
        line = lines[i].strip()
        patterns = [r'^REFERENCES?$', r'^\d+\.?\s+REFERENCES?$', r'^\d+\.?\s+References?$', r'^References?:$',
                    r'^BIBLIOGRAPHY$', r'^\d+\.?\s+BIBLIOGRAPHY$', r'^\d+\.?\s+Bibliography$', r'^Bibliography:$',
                    r'^Literature\s+Cited$', r'^Works\s+Cited$']
        if any(re.match(p, line, re.I) for p in patterns):
            following = lines[i+1:i+4]
            if any(re.search(r'\(\d{4}\)|\d{4}\.|doi:| et al', fl, re.I) for fl in following):
                cut_index = i
                break
    return '\n'.join(lines[:cut_index]).strip() if cut_index != -1 else text.strip()

# EXTRACT PDF TEXTS
def extract_text_from_pdf(pdf_path):
    article_id = Path(pdf_path).stem
    doc = fitz.open(pdf_path)
    text = "\n".join([page.get_text() for page in doc])
    doc.close()
    text = remove_references_section(text)
    return article_id, text

pdf_directory = "/kaggle/input/make-data-count-finding-data-references/test/PDF" if os.getenv('KAGGLE_IS_COMPETITION_RERUN') else "/kaggle/input/make-data-count-finding-data-references/train/PDF"
pdf_paths = [os.path.join(pdf_directory, f) for f in os.listdir(pdf_directory) if f.endswith(".pdf")]

with Pool(processes=NUM_PROCESSES) as pool:
    pdf_results = list(tqdm(pool.imap(extract_text_from_pdf, pdf_paths), total=len(pdf_paths)))

chunks, chunks2, ids = [], [], []

for article_id, text in pdf_results:
    for match in re_doi.finditer(text):
        if match.group() in article_id: continue
        chunk = text[max(0, match.start() - text_span_len): match.start() + text_span_len]
        chunks.append((article_id, chunk))

    for match in big_re.finditer(text):
        chunk = text[max(0, match.start() - text_span_len): match.start() + text_span_len]
        chunks2.append((article_id, chunk))
        ids.append(match.group())

# LLM SETUP
model_path = "/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1"
llm = vllm.LLM(model_path, quantization='awq', tensor_parallel_size=torch.cuda.device_count(),
               gpu_memory_utilization=0.9, trust_remote_code=True, dtype="half",
               enforce_eager=True, max_model_len=2048, disable_log_stats=True, enable_prefix_caching=True)
tokenizer = llm.get_tokenizer()

SYS_PROMPT_DOI = """
You are a scientific assistant.
Given a snippet of academic text, determine whether it contains a DOI referring to a research dataset.

Respond with:
- The full normalized DOI URL (e.g., https://doi.org/10.1234/abcd) if it clearly refers to research data.
- Or respond exactly with: Irrelevant

Do not output any explanation or extra content.
"""

SYS_PROMPT_CLASSIFY_DOI = """
You are a scientific assistant.
Given a DOI and a relevant academic text snippet, classify the dataset associated with the DOI as:
A) Primary — generated by this study
B) Secondary — reused from other studies
C) None — not dataset-related or just cited

Respond with exactly one letter: A, B, or C.
"""

SYS_PROMPT_ACCESSION = """
You are a scientific assistant.
Given an accession ID and an academic text snippet, classify the dataset usage as:
A) Primary — generated in this study
B) Secondary — reused from previous studies
C) None — not used or unrelated

Respond with only one letter: A, B, or C.
"""

# PROCESS PROMPTS
def process_prompts(prompts, prompt_type):
    outputs = llm.generate(
        prompts,
        vllm.SamplingParams(
            seed=777,
            temperature=0.1,
            skip_special_tokens=True,
            max_tokens=1 if prompt_type != "doi" else 32,
            logits_processors=[mclp] if prompt_type != "doi" else [],
            logprobs=3 if prompt_type != "doi" else 0,
        ),
        use_tqdm=True
    )
    if prompt_type == "doi":
        return [out.outputs[0].text.strip() for out in outputs]
    else:
        return [
            {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
            for out in outputs
        ]

# STEP 1: Extract DOIs
prompts, cache, chunks_filtered = [], {}, []
for article_id, academic_text in chunks:
    academic_text = academic_text[:PROMPT_MAX_TOKENS * 4]
    messages = [{"role": "system", "content": SYS_PROMPT_DOI}, {"role": "user", "content": academic_text}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    key = md5(prompt.encode()).hexdigest()
    if key not in cache:
        prompts.append(prompt)
        chunks_filtered.append((article_id, academic_text))
        cache[key] = None

responses = process_prompts(prompts, prompt_type="doi")
doi_pattern = re.compile(r'(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', re.I)
doi_urls = ["https://doi.org/" + doi_pattern.search(r).group(1) if (r.lower() != "irrelevant" and doi_pattern.search(r)) else "Irrelevant" for r in responses]

# STEP 2: Classify DOI Types
prompts, valid_indices = [], []
for i, (chunk, url) in enumerate(zip(chunks_filtered, doi_urls)):
    if url == "Irrelevant": continue
    article_id, academic_text = chunk
    messages = [{"role": "system", "content": SYS_PROMPT_CLASSIFY_DOI}, {"role": "user", "content": f"DOI: {url}\n\nText:\n{academic_text}"}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    prompts.append(prompt)
    valid_indices.append(i)

mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=["A", "B", "C"])
logprobs = process_prompts(prompts, prompt_type="classify")
logit_matrix = pd.DataFrame(logprobs)["A B C".split()].values
choices = ["Primary", "Secondary", None]
answers = [None] * len(chunks_filtered)
for i, pick in zip(valid_indices, np.argmax(logit_matrix, axis=1)):
    answers[i] = choices[pick]

# STEP 3: Classify Accession IDs
prompts = []
for chunk, acc_id in zip(chunks2, ids):
    article_id, academic_text = chunk
    messages = [{"role": "system", "content": SYS_PROMPT_ACCESSION}, {"role": "user", "content": f"Accession ID: {acc_id}\n\nText:\n{academic_text}"}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    prompts.append(prompt)

logprobs2 = process_prompts(prompts, prompt_type="accession")
logit_matrix2 = pd.DataFrame(logprobs2)["A B C".split()].values
answers2 = [choices[pick] for pick in np.argmax(logit_matrix2, axis=1)]

# submission
sub_df = pd.DataFrame({"article_id": [c[0] for c in chunks_filtered], "dataset_id": [x.lower() if x != "Irrelevant" else x for x in doi_urls], "type": answers})
sub_df = sub_df[sub_df["type"].notnull()].reset_index(drop=True)
sub_df2 = pd.DataFrame({"article_id": [c[0] for c in chunks2], "dataset_id": ids, "type": answers2})
sub_df2 = sub_df2[sub_df2["type"].notnull()].reset_index(drop=True)
sub_df = pd.concat([sub_df, sub_df2], ignore_index=True)
priority = {"Primary": 2, "Secondary": 1, None: 0}
sub_df['priority'] = sub_df['type'].map(priority)
sub_df = sub_df[sub_df["type"].isin(["Primary", "Secondary"])]
sub_df = sub_df.sort_values(by=["article_id", "dataset_id", "priority"], ascending=False)
sub_df = sub_df.drop_duplicates(subset=['article_id', 'dataset_id']).drop(columns='priority').reset_index(drop=True)
sub_df['row_id'] = range(len(sub_df))
sub_df.to_csv("submission.csv", index=False, columns=["row_id", "article_id", "dataset_id", "type"])

print(sub_df["type"].value_counts())

# CV
def f1_score(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) != 0 else 0.0

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    pred_df = pd.read_csv("submission.csv")
    label_df = pd.read_csv("/kaggle/input/make-data-count-finding-data-references/train_labels.csv")
    label_df = label_df[label_df['type'] != 'Missing'].reset_index(drop=True)
    hits_df = label_df.merge(pred_df, on=["article_id", "dataset_id", "type"])
    tp, fp, fn = hits_df.shape[0], pred_df.shape[0] - hits_df.shape[0], label_df.shape[0] - hits_df.shape[0]
    print("TP:", tp, "FP:", fp, "FN:", fn, "F1:", round(f1_score(tp, fp, fn), 3))